In [1]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [2]:
# Load your dataset (assuming it's in CSV format)
r = 28
os.chdir('C:\\Users\\blake\\Desktop\\AFL Odds\\Testing and Analysis\\2024 cleaned data')
match_results = pd.read_csv(f'2024 {r} afl_match_results_cleaned.csv')
os.chdir('C:\\Users\\blake\\Desktop\\AFL Odds\\python scripts\\Catagorical prediction\\LightGBM')

# Split features and target
X = match_results.drop(columns=['match.homeTeam.name', 'match.awayTeam.name','venue.name','Margin','Result'])  # Replace 'target_column' with your target column name

# Initialize LabelEncoder
encoder = LabelEncoder()
# Fit and transform the target variable
y = encoder.fit_transform(match_results['Result'])

# Mapping of classes for reference
class_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
print("Class Mapping:", class_mapping)

# Encode categorical columns
categorical_columns = X.select_dtypes(include=['object']).columns
for col in categorical_columns:
    X[col] = X[col].astype('category')

# Step 1: Split the data into train (80%) and test (20%) sets
train_size = int(len(X) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Ensure no missing or infinite values
assert not X.isnull().values.any(), "Input data contains NaN values."
assert not X.isin([np.inf, -np.inf]).values.any(), "Input data contains infinite values."

# LightGBM dataset conversion
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_test = lgb.Dataset(X_test, label=y_test, reference=lgb_train)

Class Mapping: {'BL': 0, 'BW': 1, 'D': 2, 'LL': 3, 'LW': 4}


### Run at beginning of season

In [3]:
# import time
# start_time = time.time()

# # Step 2: Time series split for training and hyperparameter tuning on the train set
# tscv = TimeSeriesSplit(n_splits=5)  # Time series split with 5 splits

# # Step 3: Initialize CatBoostClassifier
# lgb_model = lgb.LGBMClassifier(objective='multiclass')

# # Step 4: Hyperparameter tuning using GridSearchCV on the train set
# param_grid = {
#     'num_leaves': [31, 50],
#     'max_depth': [-1, 10, 20],
#     'learning_rate': [0.1, 0.01],
#     'n_estimators': [100, 200, 500],
#     'boosting_type': ['gbdt', 'dart'],
# }

# # Use GridSearchCV with time series split on the training data
# grid_search = GridSearchCV(
#     estimator=lgb_model,
#     param_grid=param_grid,
#     scoring='accuracy',
#     cv=tscv,
#     verbose=1,
#     n_jobs=-1
# )

# # Step 5: Fit GridSearchCV on the train data
# grid_search.fit(X_train, y_train)

# # Best parameters from GridSearchCV
# best_params = grid_search.best_params_
# print("Best hyperparameters:", best_params)
# print("--- %s seconds ---" % (time.time() - start_time))

### Continue programming

In [4]:
params = {
    'num_leaves': 50,
    'max_depth': -1,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'boosting_type': 'gbdt',
}

# Step 6: Train the model with the best hyperparameters on the training set (using time series splits)
final_model = lgb.LGBMClassifier(**params,objective='multiclass')


final_model.fit(X_train, y_train)

# Step 7: Test the model on the test set
y_test_pred_probs = final_model.predict_proba(X_test)  # Get the probability for each class on the test set
y_test_pred_class = np.argmax(y_test_pred_probs, axis=1)  # Class with the highest probability

# Evaluate test accuracy
accuracy = accuracy_score(y_test, y_test_pred_class)
print(f"Test Accuracy: {accuracy:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005074 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11180
[LightGBM] [Info] Number of data points in the train set: 1920, number of used features: 104
[LightGBM] [Info] Start training from score -2.014903
[LightGBM] [Info] Start training from score -1.553727
[LightGBM] [Info] Start training from score -5.075174
[LightGBM] [Info] Start training from score -1.198778
[LightGBM] [Info] Start training from score -1.057290
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

In [5]:
# Step 8: Train on the full dataset (after testing)
final_model.fit(X, y)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006420 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11314
[LightGBM] [Info] Number of data points in the train set: 2401, number of used features: 104
[LightGBM] [Info] Start training from score -2.047068
[LightGBM] [Info] Start training from score -1.575051
[LightGBM] [Info] Start training from score -4.950427
[LightGBM] [Info] Start training from score -1.200231
[LightGBM] [Info] Start training from score -1.033709
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

C:\Users\blake\anaconda3\envs\ATMenv\lib\site-packages\sklearn\utils\_tags.py:354: FutureWarning: The LGBMClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


LGBMClassifier(num_leaves=50, objective='multiclass')

In [6]:
joblib.dump(final_model, 'LightGBM_model.pkl')
with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)
with open('accuracy.pkl', 'wb') as f:
    pickle.dump(accuracy, f)
with open('round.pkl', 'wb') as f:
    pickle.dump(r, f)